In [1]:
!pip install rtdl_revisiting_models

In [2]:
# #!/usr/bin/env python3
# """
# Train an MLP model on Moltbook data using rtdl_revisiting_models.
# Uses R² as the primary evaluation metric.
# Optimized for large GPUs (A100) with memory efficiency.
# """
# import os
# # MUST be set before any CUDA operation to reduce fragmentation
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# from google.colab import drive
# drive.mount('/content/drive')

# import pandas as pd
# import numpy as np
# import torch
# import torch.nn as nn
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import r2_score
# import joblib
# import warnings
# warnings.filterwarnings('ignore')

# # Import MLP from rtdl_revisiting_models (instead of FTTransformer)
# from rtdl_revisiting_models import MLP
# from torch.cuda.amp import autocast, GradScaler
# from torch.utils.data import TensorDataset, DataLoader

# # ==========================================
# # CONFIGURATION
# # ==========================================
# DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'
# DATA_PATH = f"{DESTINATION_DIR}/moltbook_with_keyword_features.pkl"

# EMBEDDING_COL = "embeddings"
# TARGET_COL = "score"
# RANDOM_STATE = 42
# TEST_SIZE = 0.30
# VAL_SIZE_FROM_TEMP = 0.50
# DROP_COLS = ["safe_content", "content", "id"]

# # GPU configuration
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")
# if device.type == "cuda":
#     print(f"GPU: {torch.cuda.get_device_name(0)}")
#     print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
#     torch.backends.cudnn.benchmark = True

# # ==========================================
# # 1. Load and prepare data
# # ==========================================
# print("Loading data...")
# moltbook = pd.read_pickle(DATA_PATH)
# print(f"Original shape: {moltbook.shape}")

# # Expand embeddings
# embedding_lists = moltbook[EMBEDDING_COL].values
# lengths = [len(lst) for lst in embedding_lists]
# if len(set(lengths)) != 1:
#     raise ValueError("Embedding lists have varying lengths.")
# emb_dim = lengths[0]
# print(f"Embedding dimension: {emb_dim}")

# emb_df = pd.DataFrame(
#     np.vstack(embedding_lists),
#     index=moltbook.index,
#     columns=[f"emb_{i}" for i in range(emb_dim)]
# )

# # Base features and target
# X_base = moltbook.drop(columns=[TARGET_COL, EMBEDDING_COL] + DROP_COLS)
# y_raw = moltbook[TARGET_COL].clip(lower=0)
# y = np.log1p(y_raw)  # log-transform target

# # Train/val/test split
# X_base_train, X_base_temp, emb_train, emb_temp, y_train, y_temp = train_test_split(
#     X_base, emb_df, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
# )
# X_base_val, X_base_test, emb_val, emb_test, y_val, y_test = train_test_split(
#     X_base_temp, emb_temp, y_temp, test_size=VAL_SIZE_FROM_TEMP, random_state=RANDOM_STATE
# )

# X_train = pd.concat([X_base_train, emb_train], axis=1)
# X_val   = pd.concat([X_base_val, emb_val], axis=1)
# X_test  = pd.concat([X_base_test, emb_test], axis=1)

# print(f"Training set: {X_train.shape}")
# print(f"Validation set: {X_val.shape}")
# print(f"Test set: {X_test.shape}")

# # ==========================================
# # Convert to PyTorch tensors (keep on CPU initially)
# # ==========================================
# X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
# X_val_tensor   = torch.tensor(X_val.values, dtype=torch.float32)
# X_test_tensor  = torch.tensor(X_test.values, dtype=torch.float32)
# y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
# y_val_tensor   = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
# y_test_tensor  = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

# # ==========================================
# # Create DataLoaders (move batches to GPU on the fly)
# # ==========================================
# batch_size = 64          # You can adjust this value
# num_workers = 0           # Colab limitation (set to 2 if using local machine)

# train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# val_dataset   = TensorDataset(X_val_tensor, y_val_tensor)
# test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
#                           pin_memory=True, num_workers=num_workers)
# val_loader   = DataLoader(val_dataset, batch_size=batch_size*2, shuffle=False,
#                           pin_memory=True, num_workers=num_workers)
# test_loader  = DataLoader(test_dataset, batch_size=batch_size*2, shuffle=False,
#                           pin_memory=True, num_workers=num_workers)

# # ==========================================
# # 2. Define MLP model (replaces FTTransformer)
# # ==========================================
# print("\nInitializing MLP model...")
# # ==========================================
# # 2. Define MLP model (corrected parameters)
# # ==========================================
# print("\nInitializing MLP model...")
# n_num_features = X_train.shape[1]  # All features are numerical
# d_out = 1

# # Correct parameter names for rtdl_revisiting_models.MLP
# model = MLP(
#     d_in=n_num_features,          # Input dimension
#     d_out=d_out,                  # Output dimension (regression)
#     n_blocks=4,                   # Number of hidden blocks (equivalent to layers)
#     d_block=256,                  # Width of each hidden block
#     dropout=0.2                   # Dropout rate (single value for all blocks)
# )

# print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters")
# model = model.to(device)

# # ==========================================
# # 3. Training setup
# # ==========================================
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
# criterion = nn.MSELoss()
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

# # Mixed precision scaler
# scaler = GradScaler()

# n_epochs = 500
# patience = 50
# best_val_loss = float('inf')
# best_model_state = None
# patience_counter = 0

# # ==========================================
# # 4. Training loop with mixed precision and batch-level GPU transfers
# # ==========================================
# print("\nStarting training...")

# for epoch in range(n_epochs):
#     model.train()
#     epoch_loss = 0.0

#     for X_batch, y_batch in train_loader:
#         # Move batch to GPU (non_blocking for overlap)
#         X_batch = X_batch.to(device, non_blocking=True)
#         y_batch = y_batch.to(device, non_blocking=True)

#         optimizer.zero_grad()

#         # Mixed precision forward
#         with autocast():
#             predictions = model(X_batch)   # MLP only takes X (no categorical input)
#             loss = criterion(predictions, y_batch)

#         # Backward with gradient scaling
#         scaler.scale(loss).backward()
#         scaler.unscale_(optimizer)                     # for gradient clipping
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         scaler.step(optimizer)
#         scaler.update()

#         epoch_loss += loss.item() * X_batch.size(0)

#     epoch_loss /= len(train_loader.dataset)

#     # ---------- Validation ----------
#     model.eval()
#     val_loss = 0.0
#     all_preds = []
#     all_y = []
#     with torch.no_grad():
#         for X_batch, y_batch in val_loader:
#             X_batch = X_batch.to(device, non_blocking=True)
#             y_batch = y_batch.to(device, non_blocking=True)
#             with autocast():
#                 preds = model(X_batch)
#                 loss = criterion(preds, y_batch)
#             val_loss += loss.item() * X_batch.size(0)
#             all_preds.append(preds.cpu())
#             all_y.append(y_batch.cpu())

#     val_loss /= len(val_loader.dataset)
#     val_preds = torch.cat(all_preds).numpy()
#     val_true = torch.cat(all_y).numpy()
#     val_r2 = r2_score(val_true, val_preds)

#     scheduler.step(val_loss)

#     if (epoch + 1) % 20 == 0:
#         current_lr = optimizer.param_groups[0]['lr']
#         print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {epoch_loss:.6f} | "
#               f"Val Loss: {val_loss:.6f} | Val R²: {val_r2:.4f} | LR: {current_lr:.2e}")
#         if device.type == "cuda":
#             allocated = torch.cuda.memory_allocated(0) / 1e9
#             reserved = torch.cuda.memory_reserved(0) / 1e9
#             print(f"  GPU Memory: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

#     # Early stopping
#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
#         patience_counter = 0
#     else:
#         patience_counter += 1
#         if patience_counter >= patience:
#             print(f"\nEarly stopping triggered at epoch {epoch+1}")
#             break

# # Load best model
# model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})

# # ==========================================
# # 5. Final evaluation on test set
# # ==========================================
# model.eval()
# test_loss = 0.0
# all_test_preds = []
# all_test_y = []
# with torch.no_grad():
#     for X_batch, y_batch in test_loader:
#         X_batch = X_batch.to(device, non_blocking=True)
#         y_batch = y_batch.to(device, non_blocking=True)
#         with autocast():
#             preds = model(X_batch)
#             loss = criterion(preds, y_batch)
#         test_loss += loss.item() * X_batch.size(0)
#         all_test_preds.append(preds.cpu())
#         all_test_y.append(y_batch.cpu())

# test_loss /= len(test_loader.dataset)
# test_preds = torch.cat(all_test_preds).numpy()
# test_true = torch.cat(all_test_y).numpy()
# test_r2 = r2_score(test_true, test_preds)

# # Also compute final validation R² using the best model
# val_loss_final = 0.0
# all_val_preds = []
# all_val_y = []
# with torch.no_grad():
#     for X_batch, y_batch in val_loader:
#         X_batch = X_batch.to(device, non_blocking=True)
#         y_batch = y_batch.to(device, non_blocking=True)
#         with autocast():
#             preds = model(X_batch)
#         all_val_preds.append(preds.cpu())
#         all_val_y.append(y_batch.cpu())
# val_preds_final = torch.cat(all_val_preds).numpy()
# val_true_final = torch.cat(all_val_y).numpy()
# val_r2_final = r2_score(val_true_final, val_preds_final)

# print("\n" + "="*50)
# print("FINAL RESULTS")
# print("="*50)
# print(f"Validation R²: {val_r2_final:.4f}")
# print(f"Test R²: {test_r2:.4f}")
# print(f"Test MSE: {test_loss:.6f}")

# # Clear cache before saving
# torch.cuda.empty_cache()

# # ==========================================
# # 6. Save model and results
# # ==========================================
# model_info = {
#     'model_state_dict': {k: v.cpu() for k, v in best_model_state.items()},
#     'feature_names': list(X_train.columns),
#     'device_info': str(device)
# }
# torch.save(model_info, 'mlp_model.pth')
# print("\nModel saved as 'mlp_model.pth'")

# results = {
#     'r2_validation': val_r2_final,
#     'r2_test': test_r2,
#     'test_mse': test_loss,
#     'device_used': str(device)
# }
# joblib.dump(results, 'mlp_results.pkl')
# print("Results saved as 'mlp_results.pkl'")

# # Final cleanup
# if device.type == "cuda":
#     torch.cuda.empty_cache()
#     print(f"Final GPU memory allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

# print("\nTraining complete!")

In [3]:
#!/usr/bin/env python3
"""
Train an MLP model on Moltbook data using rtdl_revisiting_models.
Uses R² as the primary evaluation metric.
Optimized for large GPUs (A100) with memory efficiency.
"""
import os
# MUST be set before any CUDA operation to reduce fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Import MLP from rtdl_revisiting_models (instead of FTTransformer)
from rtdl_revisiting_models import MLP
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import TensorDataset, DataLoader

# ==========================================
# CONFIGURATION
# ==========================================
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'
DATA_PATH = f"{DESTINATION_DIR}/moltbook_with_keyword_features.pkl"

EMBEDDING_COL = "embeddings"
TARGET_COL = "score"
RANDOM_STATE = 42
TEST_SIZE = 0.30
VAL_SIZE_FROM_TEMP = 0.50
DROP_COLS = ["safe_content", "content", "id"]

# GPU configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.backends.cudnn.benchmark = True

# ==========================================
# 1. Load and prepare data
# ==========================================
print("Loading data...")
moltbook = pd.read_pickle(DATA_PATH)
print(f"Original shape: {moltbook.shape}")

# Expand embeddings
embedding_lists = moltbook[EMBEDDING_COL].values
lengths = [len(lst) for lst in embedding_lists]
if len(set(lengths)) != 1:
    raise ValueError("Embedding lists have varying lengths.")
emb_dim = lengths[0]
print(f"Embedding dimension: {emb_dim}")

emb_df = pd.DataFrame(
    np.vstack(embedding_lists),
    index=moltbook.index,
    columns=[f"emb_{i}" for i in range(emb_dim)]
)

KEPT_FEATURES = [
    "comment_existence",
    "max_early_sentiment", 
    "avg_early_sentiment",
    "min_early_sentiment",
    "punctuation_density",
    "ttr",
    "hour",
    "has_biological_tax",
    "has_lobster",
    "has_great_lobster"
]
available_features = [f for f in KEPT_FEATURES if f in moltbook.columns]
X_base = moltbook[available_features].copy()


# # Base features and target
# X_base = moltbook.drop(columns=[TARGET_COL, EMBEDDING_COL] + DROP_COLS)
y_raw = moltbook[TARGET_COL].clip(lower=0)
y = np.log1p(y_raw)  # log-transform target

# Train/val/test split
X_base_train, X_base_temp, emb_train, emb_temp, y_train, y_temp = train_test_split(
    X_base, emb_df, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_base_val, X_base_test, emb_val, emb_test, y_val, y_test = train_test_split(
    X_base_temp, emb_temp, y_temp, test_size=VAL_SIZE_FROM_TEMP, random_state=RANDOM_STATE
)

X_train = pd.concat([X_base_train, emb_train], axis=1)
X_val   = pd.concat([X_base_val, emb_val], axis=1)
X_test  = pd.concat([X_base_test, emb_test], axis=1)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

# ==========================================
# Convert to PyTorch tensors (keep on CPU initially)
# ==========================================
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val.values, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val_tensor   = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

# ==========================================
# Create DataLoaders 
# ==========================================
batch_size = 64          
num_workers = 0          

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          pin_memory=True, num_workers=num_workers)
val_loader   = DataLoader(val_dataset, batch_size=batch_size*2, shuffle=False,
                          pin_memory=True, num_workers=num_workers)
test_loader  = DataLoader(test_dataset, batch_size=batch_size*2, shuffle=False,
                          pin_memory=True, num_workers=num_workers)

# ==========================================
# 2. Define MLP model (replaces FTTransformer)
# ==========================================
print("\nInitializing MLP model...")
# ==========================================
# 2. Define MLP model (corrected parameters)
# ==========================================
print("\nInitializing MLP model...")
n_num_features = X_train.shape[1]  # All features are numerical
d_out = 1

# Correct parameter names for rtdl_revisiting_models.MLP
model = MLP(
    d_in=n_num_features,          # Input dimension
    d_out=d_out,                  # Output dimension (regression)
    n_blocks=4,                   # Number of hidden blocks (equivalent to layers)
    d_block=256,                  # Width of each hidden block
    dropout=0.2                   # Dropout rate (single value for all blocks)
)

print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters")
model = model.to(device)

# ==========================================
# 3. Training setup
# ==========================================
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

# Mixed precision scaler
scaler = GradScaler()

n_epochs = 500
patience = 50
best_val_loss = float('inf')
best_model_state = None
patience_counter = 0

# ==========================================
# 4. Training loop with mixed precision and batch-level GPU transfers
# ==========================================
print("\nStarting training...")

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to GPU (non_blocking for overlap)
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        # Mixed precision forward
        with autocast():
            predictions = model(X_batch)   # MLP only takes X (no categorical input)
            loss = criterion(predictions, y_batch)

        # Backward with gradient scaling
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                     # for gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item() * X_batch.size(0)

    epoch_loss /= len(train_loader.dataset)

    # ---------- Validation ----------
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_y = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            with autocast():
                preds = model(X_batch)
                loss = criterion(preds, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            all_preds.append(preds.cpu())
            all_y.append(y_batch.cpu())

    val_loss /= len(val_loader.dataset)
    val_preds = torch.cat(all_preds).numpy()
    val_true = torch.cat(all_y).numpy()
    val_r2 = r2_score(val_true, val_preds)

    scheduler.step(val_loss)

    if (epoch + 1) % 20 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {epoch_loss:.6f} | "
              f"Val Loss: {val_loss:.6f} | Val R²: {val_r2:.4f} | LR: {current_lr:.2e}")
        if device.type == "cuda":
            allocated = torch.cuda.memory_allocated(0) / 1e9
            reserved = torch.cuda.memory_reserved(0) / 1e9
            print(f"  GPU Memory: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

# Load best model
model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})

# ==========================================
# 5. Final evaluation on test set
# ==========================================
model.eval()
test_loss = 0.0
all_test_preds = []
all_test_y = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)
        with autocast():
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
        test_loss += loss.item() * X_batch.size(0)
        all_test_preds.append(preds.cpu())
        all_test_y.append(y_batch.cpu())

test_loss /= len(test_loader.dataset)
test_preds = torch.cat(all_test_preds).numpy()
test_true = torch.cat(all_test_y).numpy()
test_r2 = r2_score(test_true, test_preds)

# Also compute final validation R² using the best model
val_loss_final = 0.0
all_val_preds = []
all_val_y = []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)
        with autocast():
            preds = model(X_batch)
        all_val_preds.append(preds.cpu())
        all_val_y.append(y_batch.cpu())
val_preds_final = torch.cat(all_val_preds).numpy()
val_true_final = torch.cat(all_val_y).numpy()
val_r2_final = r2_score(val_true_final, val_preds_final)

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
print(f"Validation R²: {val_r2_final:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Test MSE: {test_loss:.6f}")

# Clear cache before saving
torch.cuda.empty_cache()

# ==========================================
# 6. Save model and results
# ==========================================
model_info = {
    'model_state_dict': {k: v.cpu() for k, v in best_model_state.items()},
    'feature_names': list(X_train.columns),
    'device_info': str(device)
}
torch.save(model_info, 'mlp_model.pth')
print("\nModel saved as 'mlp_model.pth'")

results = {
    'r2_validation': val_r2_final,
    'r2_test': test_r2,
    'test_mse': test_loss,
    'device_used': str(device)
}
joblib.dump(results, 'mlp_results.pkl')
print("Results saved as 'mlp_results.pkl'")

# Final cleanup
if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"Final GPU memory allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

print("\nTraining complete!")

Mounted at /content/drive
Using device: cuda
GPU: NVIDIA L4
Total GPU memory: 23.66 GB
Loading data...
Original shape: (20287, 51)
Embedding dimension: 768
Training set: (14200, 778)
Validation set: (3043, 778)
Test set: (3044, 778)

Initializing MLP model...

Initializing MLP model...
Model has 397,057 trainable parameters

Starting training...
Epoch 20/500 | Train Loss: 0.307857 | Val Loss: 0.373562 | Val R²: 0.3374 | LR: 1.00e-03
  GPU Memory: 0.02 GB allocated, 0.03 GB reserved
Epoch 40/500 | Train Loss: 0.249600 | Val Loss: 0.377597 | Val R²: 0.3302 | LR: 5.00e-04
  GPU Memory: 0.02 GB allocated, 0.03 GB reserved
Epoch 60/500 | Train Loss: 0.212373 | Val Loss: 0.382199 | Val R²: 0.3220 | LR: 1.25e-04
  GPU Memory: 0.02 GB allocated, 0.03 GB reserved

Early stopping triggered at epoch 75

FINAL RESULTS
Validation R²: 0.3541
Test R²: 0.3594
Test MSE: 0.332735

Model saved as 'mlp_model.pth'
Results saved as 'mlp_results.pkl'
Final GPU memory allocated: 0.02 GB

Training complete!
